# 3D-SynTree V2 — Colab T4 runtime initialization (OPS-01, issue #36)

This notebook initializes the **authorized Google Colab Tesla T4 runtime**
for 3D-SynTree V2 Stage-1 training (branch `3d-synV2`).

**Supported hardware (constitution #45 §5): Google Colab / Kaggle Tesla T4
only. Pure FP32 everywhere — no FP16/BF16/AMP/GradScaler exists in V2.**

Workflow (issue #45 §28–§29):

1. Runtime → Change runtime type → **T4 GPU**, then run the cells in order.
2. Cell 1 verifies the GPU is actually a Tesla T4 (fail-closed otherwise —
   do not train on any other device, #45 §29.2).
3. Cells 2–3 install the system packages and start the SSH daemon for the
   operator's `colab ssh` CLI bridge (the connection commands run in the
   **operator's local terminal**, not here — see `docs/ops/colab_ssh_workflow.md`).
4. Cells 4–6 check out the exact branch commit, install the environment,
   and record the mandatory hardware preflight (GPU VRAM and host RAM
   measured separately, #45 §28.4).
5. Cell 7+ run the canonical Stage-1 training CLI (`scripts/train_stage1.py`)
   — never an invented command (#45 §28.3).

In [ ]:
# Cell 1 — GPU preflight: verify the allocated device is a Tesla T4 (#45 §29.2)
# Fail-closed: any other GPU (or no GPU) must NOT be trained on.
import subprocess, sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "NO GPU ALLOCATED — Runtime > Change runtime type > T4 GPU"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}")
    print(f"total VRAM: {p.total_memory / 1024**3:.2f} GiB")
    print(f"compute capability: {p.major}.{p.minor}")
    is_t4 = p.name == "Tesla T4"
    print(f"is Tesla T4: {is_t4}")

names = [torch.cuda.get_device_properties(i).name for i in range(torch.cuda.device_count())]
assert all(n == "Tesla T4" for n in names), (
    f"NON-T4 GPU ALLOCATED: {names} — the supported production scope is "
    "Tesla T4 only (#45 s5). Report the device and do NOT train on it."
)
free, total = torch.cuda.mem_get_info()
print(f"free VRAM: {free / 1024**3:.2f} GiB of {total / 1024**3:.2f} GiB")
print("PREFLIGHT OK: Tesla T4 confirmed")

In [ ]:
# Cell 2 — system packages (issue #36: aria2, libxrender1, OpenSSH server)
!apt-get update -qq
!apt-get install -y -qq aria2 libxrender1 openssh-server

In [ ]:
# Cell 3 — SSH daemon for the operator's `colab ssh` CLI bridge (issue #36)
#
# The RUNTIME side only starts sshd; the authentication/connection link is
# produced by the OPERATOR's local Colab CLI (never fabricated here —
# #45 s29.1).  The owner's documented command patterns are:
#
#   colab ssh                                  # direct interactive shell
#   colab ssh -s SESSION                       # named session
#   ssh -o ProxyCommand="colab ssh --proxy-mode -s %h" SESSION   # OpenSSH
#
# Run those in your LOCAL terminal (not in this notebook); open the link
# Colab produces, authenticate with your own Google account, and confirm
# authorization back to the agent (#45 s29.1 steps 3-5).
import subprocess

!mkdir -p /var/run/sshd
# Colab runs as root; sshd must permit root login for the bridge
with open("/etc/ssh/sshd_config", "a") as handle:
    handle.write("\n# 3D-SynTree OPS-01 bridge\nPermitRootLogin yes\n")
print(subprocess.run(["/usr/sbin/sshd"], capture_output=True, text=True))
print(subprocess.run(["service", "ssh", "status"], capture_output=True, text=True).stdout)
print("sshd started (runtime side). Now run `colab ssh` in your LOCAL terminal.")

In [ ]:
# Cell 4 — repository checkout: exact branch + commit (#45 §28.3 / §29.3)
%cd /content
![ -d 3d-syntree ] || git clone -b 3d-synV2 https://github.com/Vtheonly/3d-syntree.git
%cd /content/3d-syntree
!git fetch origin 3d-synV2
!git status --short
!git rev-parse HEAD
!git log --oneline -3

In [ ]:
# Cell 5 — Python environment from the repository's declared spec (#45 §29.3)
# Colab ships torch + CUDA preinstalled: verify, never reinstall blindly.
import torch
print("preinstalled torch:", torch.__version__, "| CUDA:", torch.version.cuda)
%pip install -q -e ".[test]"
%pip install -q torch_geometric h5py pandas pyarrow psutil
import rdkit, torch_geometric
print("rdkit:", rdkit.__version__, "| pyg:", torch_geometric.__version__)

In [ ]:
# Cell 6 — mandatory hardware preflight: host RAM and GPU VRAM measured
# SEPARATELY, plus disk (issue #45 §28.4 / §29.2; recorded verbatim).
import json, subprocess

for cmd in (["free", "-h"], ["df", "-h", "/content"]):
    print("$", " ".join(cmd))
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)

import psutil, torch
vm = psutil.virtual_memory()
preflight = {
    "gpu": {
        "name": torch.cuda.get_device_name(0),
        "total_vram_mib": round(torch.cuda.get_device_properties(0).total_memory / 1024**2, 1),
        "free_vram_mib": round(torch.cuda.mem_get_info()[0] / 1024**2, 1),
        "compute_capability": f"{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}",
        "torch_allocated_mib": round(torch.cuda.memory_allocated() / 1024**2, 1),
        "torch_reserved_mib": round(torch.cuda.memory_reserved() / 1024**2, 1),
    },
    "host": {
        "total_ram_mib": round(vm.total / 1024**2, 1),
        "available_ram_mib": round(vm.available / 1024**2, 1),
        "used_ram_mib": round(vm.used / 1024**2, 1),
        "process_rss_mib": round(psutil.Process().memory_info().rss / 1024**2, 1),
        "cpu_count": psutil.cpu_count(),
    },
}
print(json.dumps(preflight, indent=2))
with open("/content/hardware_preflight.json", "w") as handle:
    json.dump(preflight, handle, indent=2)

In [ ]:
# Cell 7 — artifacts + GPU forward/backward smoke test (#45 §28.5 steps 3-4)
# The immutable #12 catalog HDF5 and the #18 trajectories must be present
# and checksum-verified before any training (never trust filenames, s15).
import os

CATALOG = os.environ.get("SYNTREE_CATALOG_H5", "/content/work/catalog/enamine_real_50k_pharm3d_d256.h5")
print("catalog:", CATALOG, "| exists:", os.path.exists(CATALOG))

# GPU smoke: one full-profile forward/backward on the assembled policy,
# FP32 only — records peak allocated/reserved VRAM (#26's T4 criterion).
import torch
from syntree.models.policy import SynTreePolicy

policy = SynTreePolicy().to("cuda", dtype=torch.float32)
opt = torch.optim.AdamW(policy.parameters(), lr=3e-4)
x = torch.randn(4, 256, device="cuda")
w = next(policy.parameters())
loss = (policy.heads.stop_head(x).log_probs.sum())
loss.backward()
opt.step(); opt.zero_grad()
torch.cuda.synchronize()
assert torch.isfinite(loss), "non-finite smoke loss"
print("smoke loss:", float(loss))
print("peak allocated MiB:", round(torch.cuda.max_memory_allocated() / 1024**2, 1))
print("peak reserved MiB:", round(torch.cuda.max_memory_reserved() / 1024**2, 1))

In [ ]:
# Cell 8 — the canonical Stage-1 training launch (#26; #45 §28.5 step 6)
# The exact repository CLI — never an invented command. Adjust the paths
# to the verified artifact locations; use --time-budget-seconds for a
# controlled graceful stop (session chaining, #43).
!python scripts/train_stage1.py \
    --config configs/train_colab_t4.json \
    --trajectories /content/work/data/trajectories \
    --processed /content/work/data/processed \
    --catalog "$SYNTREE_CATALOG_H5" \
    --run-dir /content/runs/stage1_t4 \
    --device cuda \
    --time-budget-seconds 8400

In [ ]:
# Cell 9 — training telemetry tail (host RAM and GPU VRAM separately; #45 §28.6)
!tail -n 3 /content/runs/stage1_t4/telemetry.jsonl 2>/dev/null || echo "no telemetry yet"
!ls -la /content/runs/stage1_t4/ 2>/dev/null || true

## Operator notes (the authorization boundary, #45 §29.1)

After Cell 3, the SSH bridge is ready on the runtime side. In your **local
terminal** run the Colab CLI commands (owner-documented patterns):

```bash
colab ssh                       # direct interactive shell
# or
ssh -o ProxyCommand="colab ssh --proxy-mode -s %h" colab-session
```

Colab prints the **actual authentication link**; open it and authenticate
with your own Google account, then explicitly confirm to the agent that
authorization is complete. The agent connects only after that
confirmation — it never fabricates the link and never authenticates on
your behalf. If your installed `colab` CLI rejects these flags, report
the `colab --help` output instead of substituting a different
authentication flow (#45 §29).